In [209]:
import pandas as pd
import numpy as np
import nltk
nltk.download('wordnet')
import re
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split

[nltk_data] Downloading package wordnet to /Users/paul/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [210]:
! pip install bs4 # in case you don't have it installed

# Dataset: https://s3.amazonaws.com/amazon-reviews-pds/tsv/amazon_reviews_us_Beauty_v1_00.tsv.gz
#          https://web.archive.org/web/20201127142707if_/https://s3.amazonaws.com/amazon-reviews-pds/tsv/amazon_reviews_us_Office_Products_v1_00.tsv.gz


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: pip install --upgrade pip


# Dataset Preparation


## Read Data


In [211]:
filepath = "data.tsv"

df = pd.read_csv(filepath, sep='\t', engine="python", on_bad_lines="skip")

print(df.head())
print(df.shape)

  marketplace  customer_id       review_id  product_id  product_parent  \
0          US     43081963  R18RVCKGH1SSI9  B001BM2MAC       307809868   
1          US     10951564  R3L4L6LW1PUOFY  B00DZYEXPQ        75004341   
2          US     21143145  R2J8AWXWTDX2TF  B00RTMUHDW       529689027   
3          US     52782374  R1PR37BR7G3M6A  B00D7H8XB6       868449945   
4          US     24045652  R3BDDDZMZBZDPU  B001XCWP34        33521401   

                                       product_title product_category  \
0     Scotch Cushion Wrap 7961, 12 Inches x 100 Feet  Office Products   
1          Dust-Off Compressed Gas Duster, Pack of 4  Office Products   
2  Amram Tagger Standard Tag Attaching Tagging Gu...  Office Products   
3  AmazonBasics 12-Sheet High-Security Micro-Cut ...  Office Products   
4  Derwent Colored Pencils, Inktense Ink Pencils,...  Office Products   

   star_rating  helpful_votes  total_votes vine verified_purchase  \
0            5              0            0    N

## Keep Reviews and Ratings


In [212]:
cols = ["review_body", "star_rating"]
df = df[cols]

In [213]:
with pd.option_context('display.max_colwidth', None):
    print(df.iloc[0])

review_body    Great product.
star_rating                 5
Name: 0, dtype: object


In [214]:
with pd.option_context('display.max_colwidth', None):
    print(df.iloc[4])

review_body    Gorgeous colors and easy to use
star_rating                                  4
Name: 4, dtype: object


In [215]:
with pd.option_context('display.max_colwidth', None):
    print(df.iloc[6])

review_body    Gold plated fusers are the best! It will never fail on you
star_rating                                                             5
Name: 6, dtype: object


In [216]:
# 1a) rating statistics
rating_counts = df['star_rating'].value_counts()
print(rating_counts)

star_rating
5    1580941
4     417975
1     306576
3     193452
2     138215
Name: count, dtype: int64


## Relabeling and Sampling

First form three classes and print their statistics. Then randomly select 100,000 reviews from the positive and 100,000 reviews from the negative


In [217]:
# create binary labels
df["sentiment"] = -1
df.loc[df["star_rating"] >= 4, "sentiment"] = 1
df.loc[df["star_rating"] <= 2, "sentiment"] = 0

# count sentiment classes
sentiment_counts = df["sentiment"].value_counts()
print(sentiment_counts)


sentiment
 1    1998916
 0     444791
-1     193452
Name: count, dtype: int64


In [218]:
# discard rating 3 reviews (-1 sentiment)
df = df[df["sentiment"] != -1]
sentiment_counts = df["sentiment"].value_counts()
print(sentiment_counts)
print(df.columns)

sentiment
1    1998916
0     444791
Name: count, dtype: int64
Index(['review_body', 'star_rating', 'sentiment'], dtype='str')


In [219]:
# randomly select 100,000 positive and 100,000 negative sentiment reviews
n = 100000

df = (
    df.groupby("sentiment", group_keys=False)
    .sample(n=n, random_state=42)
)

print(df["sentiment"].value_counts())

sentiment
0    100000
1    100000
Name: count, dtype: int64


# Data Cleaning


In [220]:
# print average length of reviews by char length BEFORE cleaning
avg_len = df["review_body"].str.len().mean()
print("Avg len before cleaning:", round(avg_len))

# print 3 sample reviews BEFORE cleaning and preprocessing
with pd.option_context('display.max_colwidth', None):
    print(df["review_body"][:3])

# convert all reviews to lowercase
df["review_body"] = df["review_body"].str.lower()

def strip_html_and_urls(text):
    if pd.isna(text):
        return text
    # Remove HTML
    text = BeautifulSoup(text, "html.parser").get_text()
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    return text

# remove extra spaces
df["review_body"] = df["review_body"].str.replace(r"\s+", " ", regex=True)

# perform contractions
contractions = {
    "won't": "will not",
    "wouldn't": "would not",
    "can't": "cannot",
    "couldn't": "could not",
    "i'm": "i am",
    "ain't": "am not",
    "it's": "it is",
    "you're": "you are",
    "they're": "they are",
    "i've": "i have",
    "isn't": "is not",
    "aren't": "are not",
    "don't": "do not",
    "didn't": "did not",
    "doesn't": "does not",
    "musn't": "must not",
    "hasn't": "has not",
    "hadn't": "had not",
    "haven't": "have not"
}

# add the lazy version of contractions (without apostrophe)
# to dict
for k in list(contractions.keys()):
    contractions[k.replace("'", "")] = contractions[k]

def expand_contractions(text):
    if pd.isna(text):
        return text
    pattern = re.compile(r'\b(' + '|'.join(contractions.keys()) + r')\b')
    return pattern.sub(lambda x: contractions[x.group()], text)

df["review_body"] = df["review_body"].apply(expand_contractions)

# remove non-alphabetical characters
df["review_body"] = df["review_body"].str.replace(r"[^a-zA-Z\s]", "", regex=True)

# print average length of reviews by char length AFTER cleaning
avg_len = df["review_body"].str.len().mean()
print("Avg len after cleaning:", round(avg_len))

Avg len before cleaning: 319
1862138                                                                                                                                                                                                                                                                                                                                                                                                 I wish I had read the reviews before I purchased them.  It didn't ruin the printer but they didn't work and I needed to get some photos printed.  I am extremely disappointed.
2468360    The day of the week is automatically populated when you enter the date.  But it is set for 2008, and there is no way to change it to the correct day of the week.  I set it on Saturday, and there was no way to get it to say Saturday.  It will only have the correct day of the week once every seven years.  I don't see any advantage to the magnetic key holders over simple hooks.  Other than you

# Pre-processing


## remove the stop words


In [221]:
from nltk.corpus import stopwords
nltk.download("stopwords")

# print average length of reviews by char length BEFORE preprocessing
avg_len = df["review_body"].str.len().mean()
print("Avg len before preprocessing:", round(avg_len))

stop_words = set(stopwords.words("english"))
words_to_keep = {
    "not", "no",
    "will", "would", "can", "could", "am", "is", "are",
    "have", "has", "had", "must", "do", "does", "did"
}
stop_words -= words_to_keep
df["review_body"] = df["review_body"].apply(
    lambda x: " ".join([word for word in x.split() if word not in stop_words])
    if isinstance(x, str) else ""
)

Avg len before preprocessing: 305


[nltk_data] Downloading package stopwords to /Users/paul/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## perform lemmatization


In [222]:
from nltk.stem import WordNetLemmatizer
nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()
df["review_body"] = df["review_body"].apply(lambda x: " ".join([lemmatizer.lemmatize(w, pos="v") for w in x.split()]))

# print 3 sample reviews AFTER cleaning and preprocessing
with pd.option_context('display.max_colwidth', None):
    print(df["review_body"][:3], '\n')

# print average length of reviews by char length AFTER preprocessing
avg_len = df["review_body"].str.len().mean()
print("Avg len after preprocessing:", round(avg_len))

[nltk_data] Downloading package wordnet to /Users/paul/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/paul/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


1862138                                                                                                                                                                                                          wish have read review purchase do not ruin printer do not work need get photos print be extremely disappoint
2468360    day week be automatically populate enter date be set be no way change correct day week set saturday no way get say saturday will have correct day week every seven years do not see advantage magnetic key holders simple hook be force put key magnetic key ring come product need one thing pocket would not buy
1229465                                                                                                                                                                                                                                                                                             last weeks no longer work
Name: review_body, dtype: str 

Avg len after 

# Bigram Feature Extraction


In [223]:
import nltk
nltk.download("punkt_tab")
from nltk.util import bigrams
from nltk.tokenize import word_tokenize
from collections import Counter
from sklearn.feature_extraction import DictVectorizer

def extract_bigrams(tokens):
    bg = bigrams(tokens)
    return Counter(["_".join(pair) for pair in bg])

df["tokens"] = df["review_body"].apply(word_tokenize)

# Convert Series of Counters → list of dicts
bigram_dicts = list(df["tokens"].apply(extract_bigrams))

vec = DictVectorizer(sparse=True)
X_sparse = vec.fit_transform(bigram_dicts)
y = df["sentiment"]

[nltk_data] Downloading package punkt_tab to /Users/paul/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [224]:
# convert to dataframe
# split dataset into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X_sparse,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


"""
total dataset size = 200,000
80% training = 160,000
20% test = 40,000
""" 
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(160000, 1640555)
(160000,)
(40000, 1640555)
(40000,)


# Perceptron


In [225]:
from sklearn.linear_model import Perceptron
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

perceptron = Perceptron(
    max_iter=1000,
    tol=1e-3,
    random_state=42
)

perceptron.fit(X_train, y_train)

,"penalty penalty: {'l2','l1','elasticnet'}, default=NoneThe penalty (aka regularization term) to be used.",None
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term if regularization isused.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with `0 <= l1_ratio <= 1`.`l1_ratio=0` corresponds to L2 penalty, `l1_ratio=1` to L1.Only used if `penalty='elasticnet'`... versionadded:: 0.24",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already centered.",True
,"max_iter max_iter: int, default=1000The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the ``fit`` method, and not the:meth:`partial_fit` method... versionadded:: 0.19",1000
,"tol tol: float or None, default=1e-3The stopping criterion. If it is not None, the iterations will stopwhen (loss > previous_loss - tol)... versionadded:: 0.19",0.001
,"shuffle shuffle: bool, default=TrueWhether or not the training data should be shuffled after each epoch.",True
,"verbose verbose: int, default=0The verbosity level.",0
,"eta0 eta0: float, default=1Constant by which the updates are multiplied.",1.0
,"n_jobs n_jobs: int, default=NoneThe number of CPUs to use to do the OVA (One Versus All, formulti-class problems) computation.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"random_state random_state: int, RandomState instance or None, default=0Used to shuffle the training data, when ``shuffle`` is set to``True``. Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary `.",42


In [226]:
def predict_and_print_metrics(model, model_name):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # train metrics
    train_accuracy  = accuracy_score(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred)
    train_recall    = recall_score(y_train, y_train_pred)
    train_f1        = f1_score(y_train, y_train_pred)

    # test metrics
    test_accuracy  = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall    = recall_score(y_test, y_test_pred)
    test_f1        = f1_score(y_test, y_test_pred)

    print(model_name + " Training Accuracy: {:.4f}".format(train_accuracy))
    print(model_name + " Training Precision: {:.4f}".format(train_precision))
    print(model_name + " Training Recall: {:.4f}".format(train_recall))
    print(model_name + " Training F1-score: {:.4f}".format(train_f1))

    print(model_name + " Testing Accuracy: {:.4f}".format(test_accuracy))
    print(model_name + " Testing Precision: {:.4f}".format(test_precision))
    print(model_name + " Testing Recall: {:.4f}".format(test_recall))
    print(model_name + " Testing F1-score: {:.4f}".format(test_f1))

predict_and_print_metrics(perceptron, "Perceptron")

Perceptron Training Accuracy: 0.9948
Perceptron Training Precision: 0.9908
Perceptron Training Recall: 0.9990
Perceptron Training F1-score: 0.9949
Perceptron Testing Accuracy: 0.8930
Perceptron Testing Precision: 0.8934
Perceptron Testing Recall: 0.8923
Perceptron Testing F1-score: 0.8929


# SVM


In [227]:
from sklearn.svm import LinearSVC

svm = LinearSVC(
    random_state=42,
    max_iter=10000
)

svm.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or bool, default=""auto""Select the algorithm to either solve the dual or primaloptimization problem. Prefer dual=False when n_samples > n_features.`dual=""auto""` will choose the value of the parameter automatically,based on the values of `n_samples`, `n_features`, `loss`, `multi_class`and `penalty`. If `n_samples` < `n_features` and optimizer supportschosen `loss`, `multi_class` and `penalty`, then dual will be set to True,otherwise it will be set to False... versionchanged:: 1.3 The `""auto""` option is added in version 1.3 and will be the default in version 1.5.",'auto'
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.For an intuitive visualization of the effects of scalingthe regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"multi_class multi_class: {'ovr', 'crammer_singer'}, default='ovr'Determines the multi-class strategy if `y` contains more thantwo classes.``""ovr""`` trains n_classes one-vs-rest classifiers, while``""crammer_singer""`` optimizes a joint objective over all classes.While `crammer_singer` is interesting from a theoretical perspectiveas it is consistent, it is seldom used in practice as it rarely leadsto better accuracy and is more expensive to compute.If ``""crammer_singer""`` is chosen, the options loss, penalty and dualwill be ignored.",'ovr'
,"fit_intercept fit_intercept: bool, default=TrueWhether or not to fit an intercept. If set to True, the feature vectoris extended to include an intercept term: `[x_1, ..., x_n, 1]`, where1 corresponds to the intercept. If set to False, no intercept will beused in calculations (i.e. data is expected to be already centered).",True
,"intercept_scaling intercept_scaling: float, default=1.0When `fit_intercept` is True, the instance vector x becomes ``[x_1,..., x_n, intercept_scaling]``, i.e. a ""synthetic"" feature with aconstant value equal to `intercept_scaling` is appended to the instancevector. The intercept becomes intercept_scaling * synthetic featureweight. Note that liblinear internally penalizes the intercept,treating it like any other term in the feature vector. To reduce theimpact of the regularization on the intercept, the `intercept_scaling`parameter can be set to a value greater than 1; the higher the value of`intercept_scaling`, the lower the impact of regularization on it.Then, the weights become `[w_x_1, ..., w_x_n,w_intercept*intercept_scaling]`, where `w_x_1, ..., w_x_n` representthe feature weights and the intercept weight is scaled by`intercept_scaling`. This scaling allows the intercept term to have adifferent regularization behavior compared to the other features.",1
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to ``class_weight[i]*C`` forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: int, default=0Enable verbose output. Note that this setting takes advantage of aper-process runtime setting in liblinear that, if enabled, may not workproperly in a multithreaded context.",0
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo rand

In [228]:
predict_and_print_metrics(svm, "SVM")

SVM Training Accuracy: 0.9952
SVM Training Precision: 0.9908
SVM Training Recall: 0.9997
SVM Training F1-score: 0.9952
SVM Testing Accuracy: 0.8837
SVM Testing Precision: 0.8639
SVM Testing Recall: 0.9109
SVM Testing F1-score: 0.8867


# Logistic Regression


In [229]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    random_state=42,
    max_iter=1000, 
    solver='liblinear'
)

logreg.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multi

In [230]:
predict_and_print_metrics(logreg, "Logistic Regression")

Logistic Regression Training Accuracy: 0.9887
Logistic Regression Training Precision: 0.9804
Logistic Regression Training Recall: 0.9975
Logistic Regression Training F1-score: 0.9888
Logistic Regression Testing Accuracy: 0.8985
Logistic Regression Testing Precision: 0.8767
Logistic Regression Testing Recall: 0.9276
Logistic Regression Testing F1-score: 0.9014


# Naive Bayes


In [231]:
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB()

nb.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [232]:
predict_and_print_metrics(nb, "Naive Bayes")

Naive Bayes Training Accuracy: 0.9457
Naive Bayes Training Precision: 0.9794
Naive Bayes Training Recall: 0.9106
Naive Bayes Training F1-score: 0.9438
Naive Bayes Testing Accuracy: 0.8688
Naive Bayes Testing Precision: 0.8870
Naive Bayes Testing Recall: 0.8452
Naive Bayes Testing F1-score: 0.8656
